# Generate Annotation Samples

This notebook mirrors `evaluations/create_samples.py` for interactive inspection. Use the script for the reproducible pipeline and adjust the paths below for local exploration.


In [ ]:
from pathlib import Path
import polars as pl


In [ ]:
# Configure these paths relative to the repository root.
REPO_ROOT = Path("..").resolve()
CLASSIFIED_DIR = REPO_ROOT / "classifier_output" / "classified.parquet"
SAMPLE_OUTPUT_DIR = REPO_ROOT / "samples"
HASH_SEED = 33
AUTO_MODULUS = 10_000
MANUAL_MODULUS = 100_000
SAMPLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


Run the next cell to generate the same sample files as the CLI script. The hash filter samples whole papers deterministically from `corpusid`.


In [ ]:
lf = pl.scan_parquet(CLASSIFIED_DIR)

auto_eval_lf = lf.filter(pl.col("corpusid").hash(seed=HASH_SEED) % AUTO_MODULUS == 0)
manual_eval_lf = lf.filter(pl.col("corpusid").hash(seed=HASH_SEED) % MANUAL_MODULUS == 0)

manual_filtered_lf = manual_eval_lf.filter(
    (pl.col("section_text").str.strip_chars() != "")
    & (~pl.col("sec_label_extended").is_in(["figure_table", "ending", "other"]))
)

auto_eval_lf.sink_parquet(SAMPLE_OUTPUT_DIR / "auto_eval_sample.parquet")
manual_filtered_lf.sink_parquet(SAMPLE_OUTPUT_DIR / "manual_eval_sample.parquet")
manual_filtered_lf.collect().write_csv(SAMPLE_OUTPUT_DIR / "manual_eval_sample.csv")


In [ ]:
# Inspect the human annotation sample.
manual_df = pl.read_parquet(SAMPLE_OUTPUT_DIR / "manual_eval_sample.parquet")
manual_df.select(["corpusid", "extracted", "sec_label_extended", "section_text"]).head().to_pandas()
